In [ ]:
import numpy as np
import pandas as pd

from dap_job_quality import BUCKET_NAME, logging
from dap_job_quality.getters import ojo_getters as ojo
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import analysis_utils

In [ ]:
def prop_table(full_data, grouping_cols = ['knowledge_domain','sector'], prop_col='FLEX_HOURS', sort_by = ['proportion'], ascending=False):
    prop_table = full_data.groupby(grouping_cols).agg({prop_col: sum, 'id': 'size'})
    prop_table[f'non_{prop_col}'] = prop_table['id'] - prop_table[prop_col]
    prop_table['proportion'] = prop_table[prop_col] / prop_table['id']
    return prop_table.sort_values(sort_by, ascending=ascending)

In [ ]:
processed_data = pd.read_parquet('s3://open-jobs-lake/job_quality/outputs/random/job_ads_prod_True_n_100000.parquet')

In [ ]:
titles = ojo.get_ojo_job_title_sample()
locations = ojo.get_ojo_location_sample()
salaries = ojo.get_ojo_salaries_sample()
skills = ojo.get_ojo_skills_sample()
lookup = get_keywords()

In [ ]:
titles

In [ ]:
processed_data = pd.merge(processed_data, lookup[['target_phrase', 'subcategory','dimension']], on='target_phrase', how='left')
processed_data.head()

In [ ]:
dimensions_wide = analysis_utils.create_wide_table(processed_data)
dimensions_wide

In [ ]:
full_data = pd.merge(titles, dimensions_wide, on='id', how='left')
full_data = pd.merge(locations, full_data, on='id', how='left')

Remove sectors with very few job adverts

In [ ]:
sector_counts = pd.DataFrame(full_data['sector'].value_counts())
sector_counts['count'].hist(bins=100)

In [ ]:
sector_counts['count'].describe()

In [ ]:
# Exclude sectors with fewer than 100 ads?
sector_counts[sector_counts['count'] > 100]['count'].hist(bins=100)

In [ ]:
sector_counts[sector_counts['count'] > 100]['count'].describe()

In [ ]:
sectors_to_remove = sector_counts[sector_counts['count'] < 100].index
full_data = full_data[~full_data['sector'].isin(sectors_to_remove)]

In [ ]:
full_data['itl_1_name'].value_counts()

In [ ]:
# Remove NI because the counts are noticeably lower
full_data = full_data[full_data['itl_1_name'] != 'Northern Ireland']
logging.info(f'N ads remaining after removing (a) underrepresented sectors and (b) Northern Irelands: {len(full_data)}')

In [ ]:
full_data['created'] = pd.to_datetime(full_data['created'])
full_data['year'] = full_data['created'].dt.year
# Check distribution by year
full_data['year'].value_counts()

In [ ]:
columns_to_replace = dimensions_wide.columns[1:] # the first column is the id
# These columns have NaN where there are *no* mentions of JQ dimensions in these job adverts
full_data[columns_to_replace] = full_data[columns_to_replace].fillna(0)
full_data.head()

In [ ]:
full_data = pd.merge(full_data, salaries, on='id', how='left')
full_data.head()

# Which dimensions of job quality are most frequent overall?

In [ ]:
full_data.columns
target_dims = ['CAREER','COMP','CONTRACT','FLEX_HOURS','FLEX_LOC','HOURS',
                         'L&D',                 'LEAVE',
                         'LOC', 'PERKS',
                      'SHIFT']

proportions = full_data[target_dims].sum() / len(full_data)
proportions = pd.DataFrame(proportions).reset_index()
proportions.columns = ['subcategory', 'proportion']
proportions

In [ ]:
perk_prop = proportions[proportions['subcategory'].isin(['CAREER', 'FLEX_HOURS', 'FLEX_LOC', 'L&D', 'PERKS'])]
perk_prop['type'] = 'perk'
non_perk_prop = proportions[~proportions['subcategory'].isin(['CAREER', 'FLEX_HOURS', 'FLEX_LOC', 'L&D', 'PERKS'])]
non_perk_prop['type'] = 'non_perk'
overall_prop = pd.concat([perk_prop, non_perk_prop])
overall_prop.to_csv('outputs/overall_proportions.csv', index=False)

# The perks most commonly offered were...

In [ ]:
perks_df = processed_data[processed_data['id'].isin(full_data['id'].unique())]
perks_df = perks_df[perks_df['subcategory']=='PERKS']
perks_df['target_phrase'] = perks_df['target_phrase'].str.lower()

perks_mapping = {'life assurance':['life insurance', 'life assurance'],
                 'referral scheme': ['refer a friend scheme', 'referral scheme', 'recommend a friend'],
                 'generic benefits': ['benefits', 'flexible benefits platform'],
                 'discounts/vouchers': ['discounts', 'shopping vouchers']}

def map_perk(phrase):
    for k, names in perks_mapping.items():
        if phrase in names:
            return k
    return phrase

perks_df['perk'] = perks_df['target_phrase'].apply(map_perk)
perks_prop = perks_df['perk'].value_counts(normalize=True)
perks_prop = pd.DataFrame(perks_prop).reset_index()
perks_prop

In [ ]:
perks_prop.to_csv('outputs/perks.csv', index=False)

# The occupations most and least likely to offer L&D were...

In [ ]:
full_data.columns

In [ ]:
sector_salaries = full_data.groupby(['sector']).agg({'min_annualised_salary': 'mean'}).reset_index()

In [ ]:
sector_l_and_d_prop = prop_table(full_data, grouping_cols = ['sector'], prop_col='L&D', sort_by = ['proportion'], ascending=False)

In [ ]:
sector_l_and_d_prop = pd.merge(sector_salaries, sector_l_and_d_prop, on=['sector'], how='left')#.to_csv('outputs/sector_l_and_d_prop.csv', index=False)

In [ ]:
sector_l_and_d_prop

In [ ]:
sector_l_and_d_prop.to_csv('outputs/sector_l_and_d_prop.csv', index=False)

In [ ]:
logging.info(f"Correlation between salary and offering L&D: {sector_l_and_d_prop['min_annualised_salary'].corr(sector_l_and_d_prop['proportion'])}")

# Best and worst sectors for l&d

In [ ]:
best_sectors_landd = sector_l_and_d_prop.sort_values('proportion', ascending=False).head(10)
best_sectors_landd['type'] = 'best'
worst_sectors_landd = sector_l_and_d_prop.sort_values('proportion', ascending=False).tail(10)
worst_sectors_landd['type'] = 'worst'
sectors_landd = pd.concat([best_sectors_landd, worst_sectors_landd])
sectors_landd['percentage'] = sectors_landd['proportion'] * 100
sectors_landd

In [ ]:
sectors_landd.to_csv('outputs/best_and_worst_sectors_landd.csv', index=False)

# Which sectors (knowledge domain) are most likely to offer flexible location?

In [ ]:
prop_flex_hours_by_sector = prop_table(full_data, grouping_cols = ['sector'], prop_col='FLEX_HOURS')
prop_flex_hours_by_sector

In [ ]:
prop_flex_hours_by_sector.head(10).to_csv('outputs/top_sectors_for_flex_hours.csv')
prop_flex_hours_by_sector.tail(10).to_csv('outputs/worst_sectors_for_flex_hours.csv')

In [ ]:
prop_career_by_sector = prop_table(full_data, grouping_cols = ['sector'], prop_col='CAREER')
prop_career_by_sector

In [ ]:
prop_career_by_sector.head(10)

In [ ]:
prop_career_by_sector.tail(10)

In [ ]:
avg_sal_by_kd = full_data.groupby(['knowledge_domain']).agg({'min_annualised_salary': 'mean'})#.sort_values('min_annualised_salary', ascending=False)
avg_sal_by_kd

In [ ]:
prop_flex_loc_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain'], prop_col='FLEX_LOC', sort_by=['proportion'], ascending=False)
prop_flex_loc_by_kd

In [ ]:
prop_flex_loc_kd_sal = pd.merge(prop_flex_loc_by_kd, avg_sal_by_kd, on='knowledge_domain', how='left')
prop_flex_loc_kd_sal['percentage'] = prop_flex_loc_kd_sal['proportion'] * 100
prop_flex_loc_kd_sal

In [ ]:
prop_flex_loc_kd_sal.to_csv('outputs/perc_flex_loc_kd_sal.csv')

In [ ]:
prop_flex_loc_kd_sal['min_annualised_salary'].median()

In [ ]:
prop_flex_loc_kd_sal['percentage'].median()

In [ ]:
prop_flex_loc_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain', 'itl_1_name'], prop_col='FLEX_LOC', sort_by=['proportion'], ascending=False)

prop_flex_loc_by_kd = prop_flex_loc_by_kd.reset_index()
prop_flex_loc_by_kd['percentage'] = prop_flex_loc_by_kd['proportion'] * 100
prop_flex_loc_by_kd.loc[prop_flex_loc_by_kd['id'] < 100, 'percentage'] = np.nan

prop_flex_loc_by_kd

In [ ]:
prop_flex_loc_kd_sal['percentage'].mean()

In [ ]:
matrix_df = prop_flex_loc_by_kd.pivot(index='knowledge_domain', columns='itl_1_name', values='percentage')
matrix_df

In [ ]:
matrix_df.to_csv('outputs/perc_flex_loc_kd_itl1.csv')

In [ ]:
prop_flex_loc_by_kd.to_csv('outputs/prop_flex_loc_by_kd.csv')

# Hospitality, Logistics, Education

In [ ]:
full_data['knowledge_domain'].unique()

In [ ]:
subset = full_data[full_data['knowledge_domain'].isin(['Education', 'Logistics And Transport', 'Hospitality And Catering'])]

In [ ]:
radar_data = subset.groupby('knowledge_domain').agg({'L&D': 'mean',
                                        'CAREER': 'mean',
                                        'FLEX_LOC': 'mean',
                                        'FLEX_HOURS': 'mean',
                                        'min_annualised_salary': 'mean'})
radar_data = radar_data.reset_index()
radar_data['L&D'] = radar_data['L&D'] * 100
radar_data['CAREER'] = radar_data['CAREER'] * 100
radar_data['FLEX_LOC'] = radar_data['FLEX_LOC'] * 100
radar_data['FLEX_HOURS'] = radar_data['FLEX_HOURS'] * 100
radar_data['min_annualised_salary'] = radar_data['min_annualised_salary'] / 1000

In [ ]:
radar_data.to_csv('outputs/radar_data.csv', index=False)
radar_data

# Which sectors are most likely to offer flexible hours?

In [ ]:
prop_flex_hr_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain'], prop_col='FLEX_HOURS', sort_by=['proportion'], ascending=False)
prop_flex_hr_by_kd

In [ ]:
prop_flex_loc_hr_sal = pd.merge(prop_flex_hr_by_kd, avg_sal_by_kd, on='knowledge_domain', how='left')
prop_flex_loc_hr_sal['percentage'] = prop_flex_loc_hr_sal['proportion'] * 100
prop_flex_loc_hr_sal

In [ ]:
prop_flex_loc_hr_sal.to_csv('outputs/perc_flex_loc_hr_sal.csv')

In [ ]:
prop_flex_loc_hr_sal['percentage'].median()

In [ ]:
prop_flex_loc_hr_sal['min_annualised_salary'].median()

In [ ]:
full_data['sector'].unique()

# Which sectors are most likely to offer L&D?

In [ ]:
prop_landd_by_kd = prop_table(full_data, grouping_cols = ['knowledge_domain'], prop_col='L&D', sort_by=['proportion'], ascending=False)
prop_landd_by_kd

In [ ]:
prop_landd_by_kd = pd.merge(prop_landd_by_kd, avg_sal_by_kd, on='knowledge_domain', how='left')
prop_landd_by_kd['percentage'] = prop_landd_by_kd['proportion'] * 100
prop_landd_by_kd

In [ ]:
prop_landd_by_kd.to_csv('outputs/perc_landd_sal.csv')

In [ ]:
prop_landd_by_kd['percentage'].median()

In [ ]:
prop_landd_by_kd['min_annualised_salary'].median()